In [2]:
import pandas as pd
import numpy as np

fy2012 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/LCA_FY2012_Q4.xlsx"

df = pd.read_excel(fy2012)

print(df.shape)
print(df.head())

(415845, 35)
      LCA_CASE_NUMBER               STATUS LCA_CASE_SUBMIT DECISION_DATE  \
0  I-200-10354-708540  CERTIFIED-WITHDRAWN      2010-12-20    2011-10-01   
1  I-200-10354-855690  CERTIFIED-WITHDRAWN      2010-12-30    2011-10-01   
2  I-200-11038-016044  CERTIFIED-WITHDRAWN      2011-02-07    2011-10-01   
3  I-200-11042-412895  CERTIFIED-WITHDRAWN      2011-02-11    2011-10-01   
4  I-200-11143-008515  CERTIFIED-WITHDRAWN      2011-05-23    2011-10-02   

  VISA_CLASS LCA_CASE_EMPLOYMENT_START_DATE LCA_CASE_EMPLOYMENT_END_DATE  \
0       H-1B                     2010-12-21                   2011-12-21   
1       H-1B                     2011-01-10                   2013-01-20   
2       H-1B                     2011-02-08                   2012-02-08   
3       H-1B                     2011-02-12                   2012-02-12   
4       H-1B                     2011-05-24                   2012-04-01   

  LCA_CASE_EMPLOYER_NAME LCA_CASE_EMPLOYER_ADDRESS LCA_CASE_EMPLOYER_CITY

In [4]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'DECISION_DATE', 'VISA_CLASS', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'LCA_CASE_WAGE_RATE_UNIT', 'FULL_TIME_POS', 'TOTAL_WORKERS', 'LCA_CASE_WORKLOC1_CITY', 'LCA_CASE_WORKLOC1_STATE', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'LCA_CASE_WORKLOC2_CITY', 'LCA_CASE_WORKLOC2_STATE', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [6]:
import pandas as pd

df = pd.read_excel(fy2012)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 4067
GOOGLE INC.                                           1695
APPLE INC.                                             958
AMAZON CORPORATE LLC                                   907
MICRON TECHNOLOGY, INC.                                405
FACEBOOK, INC.                                         384
ADVANCED MICRO DEVICES, INC.                           266
NVIDIA CORPORATION                                     194
TESLA MOTORS, INC.                                     142
AMAZON FULFILLMENT SERVICES, INC.                       73
FACEBOOK SERVICES, INC.                                 34
APPLEXUS TECHNOLOGIES LLC                               31
GOOGLE, INC.                                            21
APPLE INC                                               20
AMAZON WEB SERVICES LLC                                 15
MICRONESIAN TELECOMMUNICATIONS CORPORATION              13
KENNAMETAL, INC.                 

In [8]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    4067
Apple         958
Micron        405
Meta          384
AMD           266
NVIDIA        194
Tesla         142
Name: count, dtype: int64


In [10]:
df_clean = df.copy()

In [12]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["LCA_CASE_WORKLOC1_CITY"] = (
    df["LCA_CASE_WORKLOC1_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["LCA_CASE_WORKLOC1_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      156
Seattle      34
San Jose     28
New York     27
Name: count, dtype: int64


In [14]:
print(df["LCA_CASE_WAGE_RATE_UNIT"].value_counts())

LCA_CASE_WAGE_RATE_UNIT
Year    244
Hour      1
Name: count, dtype: int64


In [16]:
import pandas as pd

# Convert wage values to numbers
df["LCA_CASE_WAGE_RATE_FROM"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"],
    errors="coerce"
)

# Standardize wage units
df["LCA_CASE_WAGE_RATE_UNIT"] = (
    df["LCA_CASE_WAGE_RATE_UNIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["LCA_CASE_WAGE_RATE_FROM"]
    * df["LCA_CASE_WAGE_RATE_UNIT"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]].head(10))

      Company      City  LCA_CASE_WAGE_RATE_FROM LCA_CASE_WAGE_RATE_UNIT  \
289       AMD    Austin                  74464.0                    YEAR   
321       AMD    Austin                  77600.0                    YEAR   
1527      AMD    Austin                  80102.0                    YEAR   
2326   NVIDIA    Austin                  70970.0                    YEAR   
3942   Micron  San Jose                 117000.0                    YEAR   
4053      AMD    Austin                 112174.0                    YEAR   
8540   Micron  San Jose                 116000.0                    YEAR   
9963   Micron  San Jose                 128000.0                    YEAR   
12074  Micron  San Jose                 117000.0                    YEAR   
12513    Meta   Seattle                 157000.0                    YEAR   

       Annual_Wage  
289        74464.0  
321        77600.0  
1527       80102.0  
2326       70970.0  
3942      117000.0  
4053      112174.0  
8540      116000

In [18]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       245.000000
mean      99140.064122
std       36971.313565
min       51355.200000
25%       74464.000000
50%       90000.000000
75%      114000.000000
max      493000.000000
Name: Annual_Wage, dtype: float64


In [20]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [22]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                127      77106.0   85896.992126
1     Austin      Apple                 19      90000.0   93465.894737
2     Austin       Meta                  1      51355.2   51355.200000
3     Austin     NVIDIA                  9      98446.0  100012.000000
4   New York        AMD                  1      95492.8   95492.800000
5   New York      Apple                  1     148400.0  148400.000000
6   New York       Meta                  3     127500.0  118877.850000
7   New York  Microsoft                 22     125816.0  131953.521818
8   San Jose     Micron                 28     116500.0  113978.731429
9    Seattle       Meta                 24     125000.0  125166.666667
10   Seattle  Microsoft                 10      92502.0   95417.420000


In [24]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2012.csv",
    index=False
)